# Байесовский вывод: обновление знания о параметрах данными

**Проект «ИИ для учёных».** Практический блокнот к странице алгоритма «Байесовский вывод».

## Что делает метод

Байесовский вывод выражает знание о параметрах распределением вероятностей и обновляет его данными. Исходное представление о параметре (**априорное распределение**) соединяется с информацией из наблюдений, и по теореме Байеса получается уточнённое представление — **апостериорное распределение**. Из него извлекают точечные оценки, **интервалы доверия** с прямой вероятностной трактовкой и прогнозные распределения.

Метод применим, когда важно честно выразить неопределённость, учесть предыдущие знания или работать с малыми выборками. В этом блокноте мы оценим долю успехов и параметры линейной зависимости байесовским способом. Расчётное время прохождения — около пяти минут.

## Интуиция за методом

Представьте, что вы изучаете новый реагент: первые опыты дают неоднозначные результаты — срабатывает в 3 случаях из 8. Вы не можете с уверенностью сказать, каков истинный процент срабатывания. Байесовский вывод формализует то, что делает любой учёный в этой ситуации:

1. **До эксперимента** у вас есть некоторое предположение: «вероятно, где-то от 10 % до 90 %, без сильных предпочтений» — это **априорное распределение** (от лат. *a priori* — «из прежнего знания»).
2. **Эксперимент** показывает конкретные результаты — 3 успеха из 8. Это **функция правдоподобия**: она говорит, насколько хорошо каждое гипотетическое значение вероятности объясняет наблюдённые данные.
3. **После эксперимента** вы пересматриваете своё суждение, совместив предположение с фактами: «скорее всего, где-то 20–50 %» — это **апостериорное распределение** (от лат. *a posteriori* — «из последующего опыта»).

Чем больше данных, тем уже апостериорное распределение — оно «схлопывается» вокруг истинного значения. **Интервал доверия** (credible interval) — это диапазон, в котором истинный параметр лежит с указанной вероятностью (например, с вероятностью 95 %). В отличие от классического доверительного интервала, это утверждение имеет прямой смысл: «я уверен на 95 %, что истинное значение в этом диапазоне».

## 1. Установка библиотек

В среде Google Colab перечисленные библиотеки, как правило, уже установлены. Ячейка ниже гарантирует их наличие, в том числе при локальном запуске.

In [ ]:
%pip install -q scikit-learn numpy pandas matplotlib scipy

## 2. Единый стиль графиков

Все графики в блокнотах проекта оформляются в едином визуальном стиле сайта «ИИ для учёных»: общая палитра, шрифты, толщины линий и сетка. Ниже встроено содержимое стилевого шаблона `notebooks/viz_style.py` — отдельный файл загружать не нужно. Вызов `apply_viz_style()` настраивает matplotlib один раз на весь блокнот.

In [ ]:
# Единый стилевой шаблон графиков проекта «ИИ для учёных».
# Значения синхронизированы с токенами темы сайта (styles/tokens.css).
VIZ_TOKENS = {
    "background": "#ffffff",  # фон полотна
    "node_fill":  "#eef1f6",  # заливка блоков
    "node_text":  "#0e1726",  # основной текст
    "edge":       "#4d5e78",  # линии, оси, подписи
    "grid":       "#dce2ec",  # сетка координат
    "series":     ["#2563eb", "#0d9488", "#b45309", "#4d5e78"],  # цвета рядов
}
VIZ = VIZ_TOKENS


def apply_viz_style():
    """Настраивает matplotlib под единый стиль сайта."""
    import matplotlib as mpl
    from cycler import cycler
    t = VIZ_TOKENS
    mpl.rcParams.update({
        "font.family": "sans-serif",
        "font.sans-serif": ["Segoe UI", "DejaVu Sans", "Arial", "Helvetica"],
        "font.size": 12, "axes.titlesize": 15, "axes.titleweight": "semibold",
        "axes.labelsize": 13, "xtick.labelsize": 11, "ytick.labelsize": 11,
        "legend.fontsize": 11,
        "figure.facecolor": t["background"], "axes.facecolor": t["background"],
        "savefig.facecolor": t["background"], "figure.dpi": 110, "savefig.dpi": 150,
        "axes.prop_cycle": cycler(color=t["series"]),
        "text.color": t["node_text"], "axes.labelcolor": t["node_text"],
        "axes.titlecolor": t["node_text"], "axes.edgecolor": t["edge"],
        "xtick.color": t["edge"], "ytick.color": t["edge"], "axes.linewidth": 1.2,
        "axes.grid": True, "grid.color": t["grid"], "grid.linewidth": 1.0,
        "grid.linestyle": "-", "axes.axisbelow": True,
        "lines.linewidth": 2.0, "lines.markersize": 6, "patch.linewidth": 1.5,
        "axes.spines.top": False, "axes.spines.right": False,
        "legend.frameon": False,
    })


def get_palette(n=None):
    """Возвращает список цветов рядов из токенов темы."""
    series = VIZ_TOKENS["series"]
    if n is None:
        return list(series)
    return [series[i % len(series)] for i in range(n)]


apply_viz_style()
print("Единый стиль графиков подключён.")

## 3. Демонстрационные данные

Рассмотрим две задачи.

**Задача 1.** Оценка вероятности успеха по серии испытаний (например, доля положительных откликов в эксперименте). Мы также покажем «ага»-эксперимент: как апостериорное распределение сужается по мере накопления данных — от широкого начального незнания до уверенной оценки.

**Задача 2.** Байесовская линейная регрессия на синтетических данных с известными истинными параметрами — это позволяет проверить, насколько точно метод их восстанавливает, и увидеть полосу неопределённости вокруг предсказания.

In [ ]:
import numpy as np
import pandas as pd
from scipy import stats

rng = np.random.default_rng(42)

# Задача 1: серия испытаний Бернулли с истинной вероятностью успеха
true_p = 0.35
n_trials = 40
successes = int(rng.binomial(n_trials, true_p))
print(f'Задача 1: наблюдали {successes} успехов из {n_trials} '
      f'испытаний (истинное p = {true_p}).')

# Задача 2: линейная зависимость с шумом
true_a, true_b, noise = 2.0, -1.0, 1.2
x = np.linspace(-3, 3, 30)
y = true_a * x + true_b + rng.normal(scale=noise, size=x.size)
print(f'Задача 2: {x.size} точек, истинные a = {true_a}, b = {true_b}.')

## 4. Применение метода

### Задача 1. Оценка вероятности успеха

Для вероятности успеха удобно **бета-распределение**: оно сопряжено с биномиальным правдоподобием. «Сопряжённость» означает, что априор и апостериор имеют одинаковый вид (оба бета), и апостериорное распределение вычисляется аналитически — без численных методов.

Параметры бета-распределения `(a, b)` можно интерпретировать так: `a` — число «виртуальных» успехов, уже известных до эксперимента, `b` — число «виртуальных» неудач. Равномерный априор `Beta(1, 1)` означает: до эксперимента любое значение вероятности одинаково правдоподобно.

После наблюдения `s` успехов из `n` испытаний апостериорное распределение: `Beta(a + s, b + (n - s))`.

**Ниже мы покажем две вещи:**
1. Один финальный результат с 40 испытаниями.
2. Как апостериорное распределение постепенно сужается с 1 до 40 испытаний — ключевое «ага» байесовского вывода.

In [ ]:
import matplotlib.pyplot as plt

prior_a, prior_b = 1.0, 1.0           # равномерный априор: «всё одинаково правдоподобно»
post_a = prior_a + successes
post_b = prior_b + (n_trials - successes)

grid = np.linspace(0, 1, 400)
prior_pdf = stats.beta(prior_a, prior_b).pdf(grid)
post_pdf  = stats.beta(post_a, post_b).pdf(grid)
post_mean = post_a / (post_a + post_b)
ci = stats.beta(post_a, post_b).ppf([0.025, 0.975])

# --- График 1: априор и апостериор после всех 40 испытаний ---
fig, ax = plt.subplots(figsize=(9.5, 5.4))
ax.plot(grid, prior_pdf,
        color=VIZ['series'][3], label='априорное распределение (до опытов)')
ax.plot(grid, post_pdf,
        color=VIZ['series'][0], label=f'апостериорное распределение ({n_trials} опытов)')
ax.axvline(true_p, color=VIZ['series'][2], linestyle='--', label='истинное значение')
ax.fill_between(grid, post_pdf,
                where=(grid >= ci[0]) & (grid <= ci[1]),
                color=VIZ['series'][0], alpha=0.2, label='95% интервал доверия')
ax.set_xlabel('Вероятность успеха p')
ax.set_ylabel('Плотность распределения')
ax.set_title('Обновление знания о вероятности успеха (40 испытаний)')
ax.legend()
fig.tight_layout()
plt.show()

print(f'Апостериорное среднее: {post_mean:.3f}  (истинное: {true_p})')
print(f'95% интервал доверия: [{ci[0]:.3f}, {ci[1]:.3f}]')

# --- График 2: сужение апостериора по мере накопления данных ---
# Генерируем всю последовательность испытаний (seed зафиксирован)
rng_seq = np.random.default_rng(42)
all_outcomes = rng_seq.binomial(1, true_p, size=n_trials)  # 0/1 для каждого опыта

checkpoints = [1, 3, 7, 15, 40]
pal = get_palette(len(checkpoints))

fig, ax = plt.subplots(figsize=(10.0, 5.4))
for k, n_obs in enumerate(checkpoints):
    s_obs = all_outcomes[:n_obs].sum()
    pa = prior_a + s_obs
    pb = prior_b + (n_obs - s_obs)
    pdf_k = stats.beta(pa, pb).pdf(grid)
    ax.plot(grid, pdf_k, color=pal[k],
            label=f'n = {n_obs:2d}  (успехов: {s_obs})')
ax.axvline(true_p, color=VIZ['series'][2], linestyle='--', linewidth=1.5,
           label='истинное значение')
ax.set_xlabel('Вероятность успеха p')
ax.set_ylabel('Плотность распределения')
ax.set_title('Сужение апостериора по мере накопления данных')
ax.legend(fontsize=10)
fig.tight_layout()
plt.show()

**Как читать эти графики.**

*Первый график.* Горизонтальная ось — возможные значения вероятности успеха `p`. Вертикальная ось — плотность вероятности: чем выше кривая в точке, тем правдоподобнее это значение. Плоская линия (равномерный априор) означает «не знаю ничего»; после 40 испытаний кривая сузилась в горок вокруг истинного значения. Закрашенная область — 95 % интервал доверия: с вероятностью 95 % истинное `p` лежит именно здесь.

*Второй график.* Пять кривых соответствуют апостериорному распределению после 1, 3, 7, 15 и 40 испытаний. Обратите внимание: уже после 7 наблюдений распределение явно сместилось к истинному значению; после 40 — оно плотно сконцентрировано вокруг него. Это и есть «обновление знания»: каждая новая точка данных делает нашу оценку чуть точнее.

### Задача 2. Байесовская линейная регрессия

Оценим параметры линейной зависимости. Метод `BayesianRidge` из `scikit-learn` помещает **априорное распределение** на коэффициенты регрессии (предположение: коэффициенты небольшие, нет оснований ожидать очень больших значений), а затем, увидев данные, обновляет его до апостериорного.

В итоге мы получаем не просто одну прямую, а **прогнозное распределение**: в каждой точке модель даёт среднее значение прогноза и его стандартное отклонение. Полоса вокруг прогноза — это и есть **неопределённость** (мера того, насколько сильно результат может отличаться от среднего).

Обратите внимание: полоса должна быть шире там, куда данные не дотягиваются (экстраполяция).

In [ ]:
from sklearn.linear_model import BayesianRidge

# BayesianRidge подбирает гиперпараметры априорного распределения из данных.
# compute_score=True позволяет отслеживать маргинальное правдоподобие.
X = x.reshape(-1, 1)
br = BayesianRidge(compute_score=True)
br.fit(X, y)

# Прогноз на плотной сетке (включая зоны вне диапазона наблюдений)
x_grid = np.linspace(-4, 4, 200).reshape(-1, 1)
y_mean, y_std = br.predict(x_grid, return_std=True)

fig, ax = plt.subplots(figsize=(9.5, 5.4))
ax.scatter(x, y, color=VIZ['series'][3], zorder=5, label='наблюдения')
ax.plot(x_grid, y_mean,
        color=VIZ['series'][0], label='апостериорное среднее (прогноз)')
ax.fill_between(x_grid.ravel(),
                y_mean - 2 * y_std, y_mean + 2 * y_std,
                color=VIZ['series'][0], alpha=0.2,
                label='неопределённость (±2 ст. отклонения, ~95%)')
ax.plot(x_grid, true_a * x_grid.ravel() + true_b,
        color=VIZ['series'][2], linestyle='--', label='истинная зависимость')
ax.set_xlabel('Предиктор x')
ax.set_ylabel('Отклик y')
ax.set_title('Байесовская линейная регрессия: прогноз и неопределённость')
ax.legend()
fig.tight_layout()
plt.show()

print(f'Оценка наклона: {br.coef_[0]:.3f}  (истинный: {true_a})')
print(f'Оценка свободного члена: {br.intercept_:.3f}  (истинный: {true_b})')

**Как читать этот график.**

Точки — наблюдения с шумом. Синяя линия — лучшая оценка зависимости по данным (апостериорное среднее). Синяя полоса — диапазон неопределённости: если прогноз в точке `x = 0` равен `y = −1.0 ± 0.3`, это значит, что истинное значение, скорее всего, лежит между `−1.3` и `−0.7`. Пунктирная линия — истинная зависимость, которую модель не видела. Хорошо настроенная байесовская модель «захватывает» истинную линию внутрь полосы неопределённости.

## 5. Интерпретация результата

- **Априорное и апостериорное распределения**: априор отражает знание до эксперимента; после учёта данных распределение сужается вокруг правдоподобных значений. Чем больше данных — тем уже апостериор и тем меньше влияние выбора априора.
- **Интервал доверия** (credible interval): в отличие от классического «доверительного интервала», имеет прямую трактовку — «истинный параметр лежит в нём с вероятностью 95 % при принятых предположениях». Это именно то, что большинство учёных интуитивно хотят сказать, когда говорят «доверительный интервал».
- **Неопределённость** — это не слабость модели, а честная информация. Широкая полоса в регрессии говорит: «данных мало, экстраполируем — доверяйте меньше». Узкая полоса: «данных достаточно, мы уверены».
- **Полоса неопределённости в регрессии**: ширина полосы шире там, где данных мало, и за пределами наблюдённого диапазона — именно так и должно быть в честной модели.
- Выбор априора влияет на результат, особенно при малой выборке; разумно проверять чувствительность вывода к априору — запустите с разными `prior_a, prior_b` и сравните.

## 6. Попробуйте на своих данных

Замените демонстрационные данные своими.

1. Для задачи о вероятности успеха укажите число успехов и общее число испытаний.
2. Для регрессии загрузите файл с предиктором и откликом.
3. Снимите комментарии в ячейке ниже и выполните ячейки раздела 4.

In [ ]:
# --- Шаблон для своих данных ---
# Задача 1: доля успехов
# successes = 12        # число успехов
# n_trials  = 30        # общее число испытаний
#
# Задача 2: линейная регрессия
# df = pd.read_csv('my_data.csv')
# x = df['предиктор'].to_numpy()
# y = df['отклик'].to_numpy()
# Далее повторите ячейки раздела 4.

## Краткие выводы

- Байесовский вывод — это формализованный способ обновлять убеждения данными. В основе лежит три объекта: **априор** (что знали до), **правдоподобие** (что говорят данные) и **апостериор** (уточнённое знание).
- Главный результат — не точечная оценка, а **распределение**, которое честно показывает неопределённость.
- **Интервал доверия** в байесовском смысле имеет прямую трактовку: «истинный параметр лежит здесь с вероятностью 95 %».
- С ростом объёма данных апостериор сужается и результат становится устойчивее к выбору априора.
- Байесовская регрессия даёт **полосу неопределённости**: она шире там, где данных мало или нет совсем — это признак честной, а не самоуверенной модели.

## 7. Поэкспериментируйте сами

Попробуйте изменить следующие параметры и посмотрите, что изменится в графиках:

**Задача 1. Вероятность успеха:**
- Измените `n_trials` с 40 до 5, 10, 100. Как меняется ширина апостериорного распределения?
- Измените `prior_a, prior_b` на `(5, 5)` (слабый информативный априор вблизи 0.5) или `(1, 9)` (сильный априор, ожидаем маленький успех). При каком числе испытаний данные «побеждают» сильный априор?
- Измените `true_p` на 0.1 или 0.8. Замечаете ли вы, как при маленьких `n_trials` апостериор ещё «не долетел» до истинного значения?

**Задача 2. Регрессия:**
- Измените `noise` с 1.2 до 0.1 (почти нет шума) или 3.0 (сильный шум). Как меняется ширина полосы неопределённости?
- Уменьшите число точек `x = np.linspace(-3, 3, 8)` (вместо 30). Как изменится полоса вне диапазона данных?